In [1]:
!pip install deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.9 MB/s eta 0:00:00


In [2]:
import torch
from transformers import pipeline
import pandas as pd
from tabulate import tabulate
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
BASE = "/content/drive/MyDrive/MC859/process-data"

In [5]:
device = 0 if torch.cuda.is_available() else -1

BATCH_SIZE = 16

OUTPUT_PATH = f"{BASE}/results"
INPUT_FILE = f"{BASE}/texts/texts.txt"

## Default


In [ ]:
print("Carregando Modelo 1 (CardiffNLP - Sentimento)...")
pipe_sentiment = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    return_all_scores=True,
    device=device
)

print("Carregando Modelo 2 (Unitary - Toxicidade)...")
pipe_toxic_unitary = pipeline(
    "text-classification",
    model="unitary/unbiased-toxic-roberta",
    return_all_scores=True,
    device=device
)

print("Carregando Modelo 3 (CNERG - Ódio/Agressividade)...")
pipe_toxic_cnerg = pipeline(
    "text-classification",
    model="Hate-speech-CNERG/dehatebert-mono-portugese",
    return_all_scores=True,
    device=device
)

print("Carregando Modelo 4 (Facebook Dynabench R4)...")
pipe_toxic_fb = pipeline(
    "text-classification",
    model="facebook/roberta-hate-speech-dynabench-r4-target",
    return_all_scores=True,
    device=device
)

Carregando Modelo 1 (CardiffNLP - Sentimento)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: facebook/roberta-hate-speech-dynabench-r4-target
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
def score_cardiff(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_sentiment(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in res}
    return round((scores.get('positive', 0.0) * 10) + (scores.get('neutral', 0.0) * 5), 2)

def score_unitary(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_toxic_unitary(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in res}
    score_toxic = scores.get('toxicity', scores.get('toxic', 0.0))
    return round((1 - score_toxic) * 10, 2)

def score_cnerg(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_toxic_cnerg(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in res}
    score_hate = scores.get('hate', 0.0)
    return round((1 - score_hate) * 10, 2)

def score_facebook(text):
    if not text or str(text).strip() == "": return 5.0
    res = pipe_toxic_fb(text, truncation=True, max_length=512)
    scores = {item['label'].lower(): item['score'] for item in res}
    score_hate = scores.get('hate', 0.0)
    return round((1 - score_hate) * 10, 2)


with open(INPUT_FILE, "r", encoding="utf-8") as f:
    steam_texts = [line.strip() for line in f if line.strip()]

print(f"Total de textos carregados: {len(steam_texts)}")

data = []

for text in tqdm(steam_texts, desc="Progresso dos Modelos"):
    data.append({
        "Texto Original": text,
        "Texto (Resumo)": text if len(text) <= 40 else text[:37] + "...",
        "M1-Cardiff (Sentimento)": score_cardiff(text),
        "M2-Unitary (Toxicidade)": score_unitary(text),
        "M3-CNERG (Ódio)": score_cnerg(text),
        "M4-Facebook (Moderação)": score_facebook(text)
    })

df = pd.DataFrame(data)

df.to_csv("result_benchmark_models.csv", index=False, encoding="utf-8")

df_html = df.drop(columns=["Texto (Resumo)"])
df_html.to_html("result_benchmark_models.html", index=False, encoding="utf-8", classes="table table-striped")

print("\n" + "="*70 + "\n          TABELA COMPARATIVA DE MODELOS (ESCALA 0 A 10)\n" + "="*70)

df_terminal = df.drop(columns=["Texto Original"])

print(tabulate(df_terminal, headers='keys', tablefmt='grid', showindex=False))

print("\n[INFO] Os resultados completos foram salvos com sucesso em:")
print(" -> CSV: 'resultado_benchmark_modelos.csv'")
print(" -> HTML: 'resultado_benchmark_modelos.html' (Recomendado para ler os textos longos!)")

Total de textos carregados: 1968


Progresso dos Modelos: 100%|██████████| 1968/1968 [01:53<00:00, 17.27it/s]



          TABELA COMPARATIVA DE MODELOS (ESCALA 0 A 10)
+----------------------------------------------------------------------------+---------------------------+---------------------------+-------------------+---------------------------+
| Texto (Resumo)                                                             |   M1-Cardiff (Sentimento) |   M2-Unitary (Toxicidade) |   M3-CNERG (Ódio) |   M4-Facebook (Moderação) |
+============================================================================+===========================+===========================+===================+===========================+
| 🌩️                                                                         |                      2.39 |                      9.99 |             10    |                     10    |
+----------------------------------------------------------------------------+---------------------------+---------------------------+-------------------+---------------------------+
| 💗 💛 💜💙 💗 💚💜 💚 𝐇𝐀𝐕𝐄 𝐀 𝐁𝐄𝐀𝐔𝐓

## Translate


In [6]:
print("Carregando Modelo 1 (CardiffNLP XLM-R - Sentimento Multilíngue)...")
pipe_sentiment_multi = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    top_k=None, # <-- ATUALIZADO AQUI
    device=device
)

print("Carregando Modelo 2 (Unitary - Toxicidade Inglês)...")
pipe_toxic_unitary = pipeline(
    "text-classification",
    model="unitary/unbiased-toxic-roberta",
    top_k=None, # <-- ATUALIZADO AQUI
    device=device
)

print("Carregando Modelo 3 (HateXplain - Ódio/Agressividade Portugues)...")
pipe_hate_cnerg = pipeline(
    "text-classification",
    model="Hate-speech-CNERG/dehatebert-mono-portugese",
    top_k=None, # <-- ATUALIZADO AQUI
    device=device
)

print("Carregando Modelo 4 (Facebook Dynabench - Moderação Inglês)...")
pipe_toxic_fb = pipeline(
    "text-classification",
    model="facebook/roberta-hate-speech-dynabench-r4-target",
    top_k=None, # <-- ATUALIZADO AQUI
    device=device
)

print("\nCarregando Novos Modelos...")
# M5: Nota em Estrelas (Multilíngue - usa texto original)
pipe_stars = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    top_k=None,
    device=device
)

# M6: Análise de Emoções (Inglês - usa texto traduzido)
pipe_emotions = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=device
)

# M7: Zero-Shot Classifier (Multilíngue - usa texto original)
pipe_zero_shot = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device
)
# Definimos as categorias customizadas para o modelo Zero-Shot analisar
Z_CANDIDATE_LABELS = ["ofensa pessoal", "critica construtiva", "reclamacao de bug", "elogio"]

Carregando Modelo 1 (CardiffNLP XLM-R - Sentimento Multilíngue)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [7]:
# 2. Leitura dos Dados
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    steam_texts = [line.strip() for line in f if line.strip()]

print(f"Total de textos originais carregados: {len(steam_texts)}")

Total de textos originais carregados: 1968


In [ ]:
# Função auxiliar para a tradução paralela
def traduzir_texto(texto):
    try:
        traduzido = GoogleTranslator(source='auto', target='en').translate(texto)
        return traduzido if traduzido else texto
    except:
        return texto # Em caso de falha de conexão, mantém o original para não quebrar o código

print("Traduzindo textos para o inglês (Modo Turbo com Threads)...")
# Usa 20 threads simultâneas para ignorar o gargalo da rede
with ThreadPoolExecutor(max_workers=20) as executor:
    texts_en = list(tqdm(executor.map(traduzir_texto, steam_texts), total=len(steam_texts), desc="Tradução Paralela"))

Traduzindo textos para o inglês (Modo Turbo com Threads)...


Tradução Paralela: 100%|██████████| 1968/1968 [00:17<00:00, 110.44it/s]


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# =====================================================================
# 1. CARREGANDO O MODELO DE TRADUÇÃO BRASILEIRO (UNICAMP T5)
# =====================================================================
print("Carregando Modelo de Tradução (Unicamp T5)...")

# Nome exato do modelo no Hugging Face
model_name = "unicamp-dl/translation-pt-en-t5"

tokenizer_trans = AutoTokenizer.from_pretrained(model_name)
model_trans = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Configura o uso de Placa de Vídeo (GPU) ou Processador (CPU)
hf_device = "cuda:0" if device == 0 and torch.cuda.is_available() else "cpu"
model_trans = model_trans.to(hf_device)


# =====================================================================
# 2. TRADUÇÃO NATIVA EM LOTE (Unicamp T5)
# =====================================================================
print("Traduzindo textos originais para o inglês (Unicamp T5 via GPU)...")

texts_en = []

# A arquitetura T5 exige um "comando" antes da frase para saber o que fazer
prefixo_t5 = "translate Portuguese to English: "

# Loop passando pelos textos no tamanho do BATCH_SIZE
for i in tqdm(range(0, len(steam_texts), BATCH_SIZE), desc="Traduzindo Batches"):

    # Pega o lote atual e já embute o prefixo em cada frase usando List Comprehension
    lote_textos = [prefixo_t5 + texto for texto in steam_texts[i : i + BATCH_SIZE]]

    # 1. Converte o texto em tokens (cortando no limite de 512)
    inputs = tokenizer_trans(lote_textos, return_tensors="pt", padding=True, truncation=True, max_length=512).to(hf_device)

    # 2. Gera a tradução
    with torch.no_grad():
        translated_tokens = model_trans.generate(**inputs, max_length=512)

    # 3. Decodifica os números de volta para texto legível
    lote_traduzido = tokenizer_trans.batch_decode(translated_tokens, skip_special_tokens=True)
    texts_en.extend(lote_traduzido)

print(f"Tradução concluída: {len(texts_en)} textos processados.")

Carregando Modelo de Tradução (Unicamp T5)...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/756k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Traduzindo textos originais para o inglês (Unicamp T5 via GPU)...



Traduzindo Batches: 100%|██████████| 123/123 [14:29<00:00,  7.07s/it]

Tradução concluída: 1968 textos processados.


In [14]:
def handle_save_html(addr, df):
  # 1. Prepara o DataFrame para o HTML (Remove o resumo)
  df_html = df.drop(columns=["Texto (Resumo)"])

  # 2. Gera apenas a tag <table> com o Pandas usando a classe que você definiu no CSS
  tabela_html_pura = df_html.to_html(index=False, border=0, classes="dataframe")

  # 3. Monta a estrutura completa do site usando f-string
  html_completo = f"""<!DOCTYPE html>
  <html lang="pt-BR">
  <head>
      <meta charset="UTF-8">
      <meta name="viewport" content="width=device-width, initial-scale=1.0">
      <title>Benchmark de Modelos - NLP</title>
      <style>
      /* Reset básico */
      * {{
          box-sizing: border-box;
      }}

      /* Corpo da página */
      body {{
          margin: 0;
          padding: 24px;
          background-color: #0f172a;
          font-family: Arial, Helvetica, sans-serif;
          color: #e2e8f0;
      }}

      /* Container com scroll horizontal */
      .table-container {{
          width: 100%;
          overflow-x: auto;
          border-radius: 14px;
          border: 1px solid #334155;
          background-color: #111827;
          box-shadow: 0 4px 20px rgba(0,0,0,0.35);
      }}

      /* Tabela */
      table.dataframe {{
          width: 100%;
          min-width: 1000px; /* evita quebrar tudo */
          border-collapse: collapse;
          table-layout: fixed;
      }}

      /* Cabeçalho */
      table.dataframe thead {{
          position: sticky;
          top: 0;
          z-index: 2;
      }}

      table.dataframe thead th {{
          background-color: #1e293b;
          color: #f8fafc;
          padding: 14px 16px;
          font-size: 14px;
          font-weight: 600;
          text-align: center;
          border-bottom: 2px solid #475569;
          white-space: nowrap;
      }}

      /* Células */
      table.dataframe td {{
          padding: 12px 16px;
          border-bottom: 1px solid #334155;
          color: #e2e8f0;
          vertical-align: top;
          word-wrap: break-word;
          overflow-wrap: break-word;
      }}

      /* Primeira coluna maior (Texto Original) */
      table.dataframe td:first-child,
      table.dataframe th:first-child {{
          width: calc(45% / 2);
          text-align: left;
      }}

      /* Segunda coluna maior (Texto Traduzido) */
      table.dataframe td:nth-child(2),
      table.dataframe th:nth-child(2) {{
          width: calc(45% / 2);
          text-align: left;
      }}

      /* Outras colunas centralizadas */
      table.dataframe td:not(:first-child):not(:nth-child(2)),
      table.dataframe th:not(:first-child):not(:nth-child(2)) {{
          text-align: center;
          width: auto;
      }}

      /* Zebra striping */
      table.dataframe tbody tr:nth-child(even) {{
          background-color: #172033;
      }}

      table.dataframe tbody tr:nth-child(odd) {{
          background-color: #111827;
      }}

      /* Hover */
      table.dataframe tbody tr:hover {{
          background-color: #263449;
          transition: 0.2s;
      }}

      /* Scroll customizado */
      .table-container::-webkit-scrollbar {{
          height: 10px;
      }}

      .table-container::-webkit-scrollbar-track {{
          background: #0f172a;
      }}

      .table-container::-webkit-scrollbar-thumb {{
          background: #475569;
          border-radius: 10px;
      }}

      .table-container::-webkit-scrollbar-thumb:hover {{
          background: #64748b;
      }}
      </style>
  </head>
  <body>
      <div class="table-container">
          {tabela_html_pura}
      </div>
  </body>
  </html>
  """

  # 4. Salva o arquivo HTML final
  with open(addr, "w", encoding="utf-8") as f:
      f.write(html_completo)

In [15]:
# =====================================================================
# 3. INFERÊNCIA NOS MODELOS (USANDO BATCHING)
# =====================================================================

print("\n[Inferência] Rodando M1 - Sentimento (Usando Textos Originais/Multilíngues)...")
res_m1 = pipe_sentiment_multi(steam_texts, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M2 - Toxicidade (Usando Textos Traduzidos)...")
res_m2 = pipe_toxic_unitary(texts_en, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M3 - Ódio (Texto Original)...")
res_m3 = pipe_hate_cnerg(steam_texts, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M4 - Facebook Dynabench (Usando Textos Traduzidos)...")
res_m4 = pipe_toxic_fb(texts_en, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M5 - Classificação por Estrelas (Texto Original)...")
res_m5 = pipe_stars(steam_texts, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M6 - Emoções (Texto Traduzido)...")
res_m6 = pipe_emotions(texts_en, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M7 - Zero-Shot Customizado (Texto Original)...")
# O modelo zero-shot tem uma estrutura de chamada ligeiramente diferente
res_m7 = pipe_zero_shot(steam_texts, candidate_labels=Z_CANDIDATE_LABELS, batch_size=BATCH_SIZE, truncation=True, max_length=512)

# =====================================================================
# 4. PROCESSAMENTO E EXPORTAÇÃO (ATUALIZADO)
# =====================================================================

def extrair_scores(resultado_modelo):
    if isinstance(resultado_modelo, dict):
        return {resultado_modelo['label'].lower(): resultado_modelo['score']}
    elif isinstance(resultado_modelo, list):
        return {item['label'].lower(): item['score'] for item in resultado_modelo}
    return {}

data = []
for i in range(len(steam_texts)):
    # M1 (Cardiff) - Multilíngue
    scores_m1 = extrair_scores(res_m1[i])
    s_m1 = round((scores_m1.get('positive', 0.0) * 10) + (scores_m1.get('neutral', 0.0) * 5), 2)

    # M2 (Unitary) - Inglês
    scores_m2 = extrair_scores(res_m2[i])
    s_m2 = round((1 - scores_m2.get('toxicity', scores_m2.get('toxic', 0.0))) * 10, 2)

    # M3 (HateXplain) - Inglês
    scores_m3 = extrair_scores(res_m3[i])
    s_m3 = round((1 - scores_m3.get('hate speech', scores_m3.get('hate', 0.0))) * 10, 2)

    # M4 (Facebook) - Inglês
    scores_m4 = extrair_scores(res_m4[i])
    s_m4 = round((1 - scores_m4.get('hate', 0.0)) * 10, 2)

    # M5: Estrelas (Vem como '1 star', '2 stars', etc. Vamos converter para escala 0 a 10)
    scores_m5 = extrair_scores(res_m5[i])
    # Pega a label com maior score
    label_estrela = max(scores_m5, key=scores_m5.get)
    num_estrelas = int(label_estrela.split()[0]) # Extrai o número '1' de '1 star'
    s_m5 = round((num_estrelas / 5) * 10, 2) # Converte escala 1-5 para 0-10

    # M6: Emoções (Vamos extrair a emoção predominante e o score de Raiva)
    scores_m6 = extrair_scores(res_m6[i])
    emocao_predominante = max(scores_m6, key=scores_m6.get)
    score_raiva = round(scores_m6.get('anger', 0.0) * 10, 2) # Raiva mapeada de 0 a 10

    # M7: Zero-Shot (A estrutura do zero-shot retorna chaves 'labels' e 'scores' emparelhadas)
    resultado_z = res_m7[i]
    mapping_z = dict(zip(resultado_z['labels'], resultado_z['scores']))
    # Vamos capturar a probabilidade de ser uma "ofensa pessoal" e "reclamacao de bug" (escala 0 a 10)
    s_ofensa = round(mapping_z.get('ofensa pessoal', 0.0) * 10, 2)
    s_bug = round(mapping_z.get('reclamacao de bug', 0.0) * 10, 2)

    # Consolidação dos Dados
    data.append({
        "Texto Original": steam_texts[i],
        "Texto Traduzido": texts_en[i],
        "Texto (Resumo)": steam_texts[i] if len(steam_texts[i]) <= 30 else steam_texts[i][:27] + "...",
        "M1-Sentimento": s_m1,
        "M2-Toxicidade": s_m2,
        "M3-HateSpeech": s_m3,
        "M4-Facebook": s_m4,
        "M5-Estrelas(0-10)": s_m5,
        "M6-Emoção Principal": emocao_predominante.upper(),
        "M6-Score Raiva": score_raiva,
        "M7-Ofensa Pessoal": s_ofensa,
        "M7-Prob. Bug": s_bug
    })

# Criando o DataFrame completo (Contém Original e Traduzido)
df = pd.DataFrame(data)

# Salvando no CSV (Completo com as duas linguagens)
df.to_csv(f"{OUTPUT_PATH}/result_benchmark_models.csv", index=False, encoding="utf-8")

# Salvando no HTML (Remove o resumo para exibir os textos completos lado a lado)
# df_html = df.drop(columns=["Texto (Resumo)"])
# df_html.to_html(f"{OUTPUT_PATH}/result_benchmark_models.html", index=False, encoding="utf-8", classes="table table-striped")
handle_save_html(f"{OUTPUT_PATH}/result_benchmark_models_translate.html", df)

print("\n" + "="*70 + "\n          TABELA COMPARATIVA DE MODELOS (ESCALA 0 A 10)\n" + "="*70)

# Para exibição no Terminal, removemos o Original e o Traduzido longos.
# Assim, mantemos apenas o "Texto (Resumo)" para a tabela não quebrar na tela.
df_terminal = df.drop(columns=["Texto Original", "Texto Traduzido"])

print(tabulate(df_terminal, headers='keys', tablefmt='grid', showindex=False))

print("\n[INFO] Os resultados completos foram salvos com sucesso!")
print(" -> CSV: 'result_benchmark_models.csv' (Contém Texto Original e Traduzido)")
print(" -> HTML: 'result_benchmark_models.html' (Melhor para comparar as duas traduções lado a lado)")


[Inferência] Rodando M1 - Sentimento (Usando Textos Originais/Multilíngues)...
[Inferência] Rodando M2 - Toxicidade (Usando Textos Traduzidos)...
[Inferência] Rodando M3 - Ódio (Texto Original)...
[Inferência] Rodando M4 - Facebook Dynabench (Usando Textos Traduzidos)...
[Inferência] Rodando M5 - Classificação por Estrelas (Texto Original)...
[Inferência] Rodando M6 - Emoções (Texto Traduzido)...
[Inferência] Rodando M7 - Zero-Shot Customizado (Texto Original)...

          TABELA COMPARATIVA DE MODELOS (ESCALA 0 A 10)
+-----------------------------------------------------------+-----------------+-----------------+-----------------+---------------+---------------------+-----------------------+------------------+---------------------+----------------+
| Texto (Resumo)                                            |   M1-Sentimento |   M2-Toxicidade |   M3-HateSpeech |   M4-Facebook |   M5-Estrelas(0-10) | M6-Emoção Principal   |   M6-Score Raiva |   M7-Ofensa Pessoal |   M7-Prob. Bug |
+=

## Clear Text

In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.8 MB/s eta 0:00:00


In [ ]:
import re
import emoji

In [ ]:
def clear_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove emojis
    text_clear = emoji.replace_emoji(text, replace='')

    # 2. Remove caracteres de ASCII art (Box Drawing e Block Elements)
    # Isso elimina a maioria das copypastas visuais (Shrek, gatos, etc)
    text_clear = re.sub(r'[\u2500-\u259F\u2800-\u28FF]', '', text_clear)

    # 3. Reduz caracteres repetidos (ex: "ameeeei!!!" vira "ameei!!")
    text_clear = re.sub(r'(.)\1{2,}', r'\1\1', text_clear)

    # 4. Remove links (opcional, mas recomendado)
    text_clear = re.sub(r'http\S+|www.\S+', '', text_clear)

    # 5. Remove espaços em branco extras e normaliza
    text_clear = re.sub(r'\s+', ' ', text_clear).strip()

    return text_clear

In [ ]:
clear_steam_text = []
# 2. Leitura dos Dados
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        texto_limpo = clear_text(line)
        # Só adiciona se sobrou algum texto após a limpeza
        # (evita que linhas que eram SÓ emojis fiquem vazias)
        if len(texto_limpo) > 2:
            clear_steam_text.append(texto_limpo)

print(f"Total de textos originais carregados: {len(clear_steam_text)}")

Total de textos originais carregados: 1851


In [ ]:
print("Traduzindo textos para o inglês (Modo Turbo com Threads)...")
# Usa 20 threads simultâneas para ignorar o gargalo da rede
with ThreadPoolExecutor(max_workers=20) as executor:
    clear_texts_en = list(tqdm(executor.map(traduzir_texto, clear_steam_text), total=len(clear_steam_text), desc="Tradução Paralela"))

Traduzindo textos para o inglês (Modo Turbo com Threads)...


Tradução Paralela: 100%|██████████| 1851/1851 [00:29<00:00, 62.64it/s]


In [ ]:
# =====================================================================
# 3. INFERÊNCIA NOS MODELOS (USANDO BATCHING)
# =====================================================================

print("\n[Inferência] Rodando M1 - Sentimento (Usando Textos Originais/Multilíngues)...")
res_m1 = pipe_sentiment_multi(clear_steam_text, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M2 - Toxicidade (Usando Textos Traduzidos)...")
res_m2 = pipe_toxic_unitary(clear_texts_en, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M3 - Ódio (Texto Original)...")
res_m3 = pipe_hate_cnerg(clear_steam_text, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M4 - Facebook Dynabench (Usando Textos Traduzidos)...")
res_m4 = pipe_toxic_fb(clear_texts_en, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M5 - Classificação por Estrelas (Texto Original)...")
res_m5 = pipe_stars(clear_steam_text, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M6 - Emoções (Texto Traduzido)...")
res_m6 = pipe_emotions(clear_texts_en, batch_size=BATCH_SIZE, truncation=True, max_length=512)

print("[Inferência] Rodando M7 - Zero-Shot Customizado (Texto Original)...")
# O modelo zero-shot tem uma estrutura de chamada ligeiramente diferente
res_m7 = pipe_zero_shot(clear_steam_text, candidate_labels=Z_CANDIDATE_LABELS, batch_size=BATCH_SIZE, truncation=True, max_length=512)

# =====================================================================
# 4. PROCESSAMENTO E EXPORTAÇÃO (ATUALIZADO)
# =====================================================================

def extrair_scores(resultado_modelo):
    if isinstance(resultado_modelo, dict):
        return {resultado_modelo['label'].lower(): resultado_modelo['score']}
    elif isinstance(resultado_modelo, list):
        return {item['label'].lower(): item['score'] for item in resultado_modelo}
    return {}

data = []
for i in range(len(clear_steam_text)):
    # M1 (Cardiff) - Multilíngue
    scores_m1 = extrair_scores(res_m1[i])
    s_m1 = round((scores_m1.get('positive', 0.0) * 10) + (scores_m1.get('neutral', 0.0) * 5), 2)

    # M2 (Unitary) - Inglês
    scores_m2 = extrair_scores(res_m2[i])
    s_m2 = round((1 - scores_m2.get('toxicity', scores_m2.get('toxic', 0.0))) * 10, 2)

    # M3 (HateXplain) - Inglês
    scores_m3 = extrair_scores(res_m3[i])
    s_m3 = round((1 - scores_m3.get('hate speech', scores_m3.get('hate', 0.0))) * 10, 2)

    # M4 (Facebook) - Inglês
    scores_m4 = extrair_scores(res_m4[i])
    s_m4 = round((1 - scores_m4.get('hate', 0.0)) * 10, 2)

    # M5: Estrelas (Vem como '1 star', '2 stars', etc. Vamos converter para escala 0 a 10)
    scores_m5 = extrair_scores(res_m5[i])
    # Pega a label com maior score
    label_estrela = max(scores_m5, key=scores_m5.get)
    num_estrelas = int(label_estrela.split()[0]) # Extrai o número '1' de '1 star'
    s_m5 = round((num_estrelas / 5) * 10, 2) # Converte escala 1-5 para 0-10

    # M6: Emoções (Vamos extrair a emoção predominante e o score de Raiva)
    scores_m6 = extrair_scores(res_m6[i])
    emocao_predominante = max(scores_m6, key=scores_m6.get)
    score_raiva = round(scores_m6.get('anger', 0.0) * 10, 2) # Raiva mapeada de 0 a 10

    # M7: Zero-Shot (A estrutura do zero-shot retorna chaves 'labels' e 'scores' emparelhadas)
    resultado_z = res_m7[i]
    mapping_z = dict(zip(resultado_z['labels'], resultado_z['scores']))
    # Vamos capturar a probabilidade de ser uma "ofensa pessoal" e "reclamacao de bug" (escala 0 a 10)
    s_ofensa = round(mapping_z.get('ofensa pessoal', 0.0) * 10, 2)
    s_bug = round(mapping_z.get('reclamacao de bug', 0.0) * 10, 2)

    # Consolidação dos Dados
    data.append({
        "Texto Original": clear_steam_text[i],
        "Texto Traduzido": clear_texts_en[i],
        "Texto (Resumo)": clear_steam_text[i] if len(clear_steam_text[i]) <= 30 else clear_steam_text[i][:27] + "...",
        "M1-Sentimento": s_m1,
        "M2-Toxicidade": s_m2,
        "M3-HateSpeech": s_m3,
        "M4-Facebook": s_m4,
        "M5-Estrelas(0-10)": s_m5,
        "M6-Emoção Principal": emocao_predominante.upper(),
        "M6-Score Raiva": score_raiva,
        "M7-Ofensa Pessoal": s_ofensa,
        "M7-Prob. Bug": s_bug
    })

# Criando o DataFrame completo (Contém Original e Traduzido)
df = pd.DataFrame(data)

# Salvando no CSV (Completo com as duas linguagens)
df.to_csv(f"{OUTPUT_PATH}/result_benchmark_models.csv", index=False, encoding="utf-8")

# Salvando no HTML (Remove o resumo para exibir os textos completos lado a lado)
# df_html = df.drop(columns=["Texto (Resumo)"])
# df_html.to_html(f"{OUTPUT_PATH}/result_benchmark_models.html", index=False, encoding="utf-8", classes="table table-striped")
handle_save_html(f"{OUTPUT_PATH}/result_benchmark_models_clear.html", df)

print("\n" + "="*70 + "\n          TABELA COMPARATIVA DE MODELOS (ESCALA 0 A 10)\n" + "="*70)

# Para exibição no Terminal, removemos o Original e o Traduzido longos.
# Assim, mantemos apenas o "Texto (Resumo)" para a tabela não quebrar na tela.
df_terminal = df.drop(columns=["Texto Original", "Texto Traduzido"])

print(tabulate(df_terminal, headers='keys', tablefmt='grid', showindex=False))

print("\n[INFO] Os resultados completos foram salvos com sucesso!")
print(" -> CSV: 'result_benchmark_models.csv' (Contém Texto Original e Traduzido)")
print(" -> HTML: 'result_benchmark_models.html' (Melhor para comparar as duas traduções lado a lado)")


[Inferência] Rodando M1 - Sentimento (Usando Textos Originais/Multilíngues)...
[Inferência] Rodando M2 - Toxicidade (Usando Textos Traduzidos)...
[Inferência] Rodando M3 - Ódio (Texto Original)...
[Inferência] Rodando M4 - Facebook Dynabench (Usando Textos Traduzidos)...
[Inferência] Rodando M5 - Classificação por Estrelas (Texto Original)...
[Inferência] Rodando M6 - Emoções (Texto Traduzido)...
[Inferência] Rodando M7 - Zero-Shot Customizado (Texto Original)...

          TABELA COMPARATIVA DE MODELOS (ESCALA 0 A 10)
+--------------------------------------------+-----------------+-----------------+-----------------+---------------+---------------------+-----------------------+------------------+---------------------+----------------+
| Texto (Resumo)                             |   M1-Sentimento |   M2-Toxicidade |   M3-HateSpeech |   M4-Facebook |   M5-Estrelas(0-10) | M6-Emoção Principal   |   M6-Score Raiva |   M7-Ofensa Pessoal |   M7-Prob. Bug |
+===============================